# CellWhisperer Zero-shot Prediction Benchmark

Reimplements the zero-shot prediction logic from plot_confusion_matrix_and_get_performance_metrics for trained SpotWhisperer models.

In [ ]:
import warnings

# avoid DeprecationWarning
warnings.filterwarnings(
    "ignore", category=DeprecationWarning, module="pandas.core.algorithms"
)

In [ ]:
import pandas as pd
import logging
import numpy as np
import copy
import matplotlib
import torch
import sys
import seaborn as sns
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

from cellwhisperer.validation.zero_shot.functions import get_performance_metrics_transcriptome_vs_text
from cellwhisperer.utils.model_io import load_cellwhisperer_model


sys.path.append(
    Path("~/cellwhisperer_private/src/figures/notebooks").expanduser().as_posix()
)  # TODO tmp required for zero_shot_validation_scripts :()
from zero_shot_validation_scripts.utils import (
    TABSAP_WELLSTUDIED_COLORMAPPING,
    PANCREAS_ORDER,
    SUFFIX_PREFIX_DICT,
)
from zero_shot_validation_scripts.dataset_preparation import load_and_preprocess_dataset

sc.set_figure_params(vector_friendly=True, dpi_save=500)

In [ ]:
#### Parameters ####

matplotlib.style.use(snakemake.input.mpl_style)

dataset_name = snakemake.wildcards.dataset
metadata_col = snakemake.wildcards.metadata_col
model = snakemake.wildcards.model

print(f"Running CellWhisperer benchmark for:")
print(f"  Dataset: {dataset_name}")
print(f"  Metadata column: {metadata_col}")
print(f"  Model: {model}")

In [ ]:
# Load model
(
    pl_model_cellwhisperer,
    text_processor_cellwhisperer,
    cellwhisperer_transcriptome_processor,
    cellwhisperer_image_processor
) = load_cellwhisperer_model(model_path=snakemake.input.model, eval=True)
cellwhisperer_model = pl_model_cellwhisperer.model

print(f"Model loaded successfully: {type(cellwhisperer_model)}")

In [ ]:
# Load data
adata = load_and_preprocess_dataset(
    dataset_name=dataset_name,
    read_count_table_path=snakemake.input.raw_read_count_table,
    obsm_paths={
        "X_cellwhisperer": (snakemake.input.processed_dataset, "transcriptome_embeds"),
        "X_geneformer": (snakemake.input.processed_dataset, "transcriptome_features"),
    },
)
logging.info(f"Data loaded and preprocessed. Shape: {adata.shape}")

In [ ]:
texts_cat = pd.Categorical(adata.obs[metadata_col])

if snakemake.params.use_prefix_suffix_version and metadata_col in SUFFIX_PREFIX_DICT:
    prefix, suffix = SUFFIX_PREFIX_DICT[metadata_col]
    texts_cat.rename_categories(lambda x: f"{prefix}{x}{suffix}", inplace=True)

In [ ]:
# Run zero-shot prediction using the same logic as the original implementation

performance_metrics, performance_metrics_per_metadata = (
    get_performance_metrics_transcriptome_vs_text(
        modality_input=adata,
        text_list_or_text_embeds=texts_cat.categories.tolist(),
        correct_text_idx_per_transcriptome=texts_cat.codes.tolist(),
        model=cellwhisperer_model,
        average_mode="embeddings",
        grouping_keys=texts_cat.codes.tolist() 
    )
)

print(f"Performance metrics computed:")
print(f"  Accuracy: {performance_metrics.get('accuracy', 'N/A')}")
print(f"  F1 score: {performance_metrics.get('f1_weighted', 'N/A')}")

In [ ]:
performance_metrics

In [ ]:
performance_metrics_per_metadata

In [ ]:
performance_metrics_nonaveraged

In [ ]:
performance_metrics_per_metadata_nonaveraged

In [ ]:
# Save performance metrics
pd.DataFrame([performance_metrics]).to_csv(
    snakemake.output.performance_metrics, index=False
)

# Save per-metadata performance metrics
performance_metrics_per_metadata.to_csv(
    snakemake.output.performance_metrics_per_metadata, index=True
)